# OpenSLR SLR40 Zeroth-Korean — 탐색적 데이터 분석 (EDA)

`/data/ASR/RAW/OpenSLR_SLR40_Zeroth_ko`

## 목적
- 파일 구조 / 인벤토리 파악, 결측·이상치 점검
- 전사 정리, 오디오 속성(FLAC) 확인


## Zeroth 특이사항 (KsponSpeech와 다른 점)
- **LibriSpeech식 구조**: `train_data_01` / `test_data_01` 폴더로 split 분리.
- **낭독체(read speech)** → 전사가 깨끗할 가능성(b/n/l/o/u 태그·이중전사 없음). 셀에서 확인.
- **`AUDIO_INFO`에 화자 메타(성별 등)** 가 있음 ex) SPEAKERID, NAME, SEX, SCRIPTID, DATASET  -> 104|BaekGeonJong|m|003|test_data_01

In [1]:
from pathlib import Path
import re, numpy as np, pandas as pd
from collections import Counter
from IPython.display import Audio, display

try:
    import soundfile as sf
except ImportError:
    sf = None
    print("⚠ soundfile 없음 → 'pip install soundfile' 필요 (FLAC 읽기/재생용)")

ROOT  = Path("/data/ASR/RAW/OpenSLR_SLR40_Zeroth_ko")
TRAIN = ROOT / "train_data_01"
TEST  = ROOT / "test_data_01"

def flac_info(p):
    """FLAC 헤더에서 (sr, channels, subtype, duration) 직접 읽기"""
    i = sf.info(str(p))
    return i.samplerate, i.channels, i.subtype, round(i.frames / i.samplerate, 3)

def play_flac(p):
    data, sr = sf.read(str(p))
    return Audio(data, rate=sr)

print("ROOT 존재:", ROOT.exists())
for x in sorted(ROOT.iterdir()):
    tag = "(dir)" if x.is_dir() else f"({x.stat().st_size/1e6:.0f}MB)"
    print("  ", x.name, tag)

ROOT 존재: True
   AUDIO_INFO (0MB)
   test_data_01 (dir)
   train_data_01 (dir)
   zeroth.lm.fg.arpa.gz (4444MB)
   zeroth.lm.tg.arpa.gz (2804MB)
   zeroth.lm.tgmed.arpa.gz (66MB)
   zeroth.lm.tgsmall.arpa.gz (26MB)
   zeroth_korean.tar.gz (10340MB)
   zeroth_lexicon (24MB)
   zeroth_morfessor.seg (352MB)


## 1. 디렉토리 구조 / 인벤토리

In [2]:
# train_data_01 구조 들여다보기 (LibriSpeech식 reader/chapter 추정)
print("train_data_01 하위 예시:")
for sub in sorted([d for d in TRAIN.iterdir() if d.is_dir()])[:3]:
    print("  ", sub.name)
    for x in sorted(sub.iterdir())[:3]:
        print("      ", x.name, "(dir)" if x.is_dir() else "")

# trans.txt / flac 형식 확인
trans = next(TRAIN.rglob("*.trans.txt"))
print(f"\n예시 trans 파일: {trans.relative_to(ROOT)}")
for line in trans.read_text(encoding="utf-8", errors="replace").splitlines()[:3]:
    print("   ", line[:100])

flac = next(TRAIN.rglob("*.flac"))
print(f"\n예시 flac: {flac.relative_to(ROOT)}")

train_data_01 하위 예시:
   003
       106 (dir)
       107 (dir)
       108 (dir)

예시 trans 파일: train_data_01/003/181/181_003.trans.txt
    181_003_0072 탈북자 열 세 명이 머무는 북한이탈주민보호센터는 탈북자가 국내에 들어와 처음 가는 곳이다
    181_003_0133 비문계 의원들에게 욕설의 의미가 담긴 십 팔 원 후원금이 후원 계좌에 입금되는 사례도 있었다
    181_003_0189 제주로 오기 전 직업은 사무 관리직이 가장 많고 자영업 판매 서비스직 순이나 이주 후 직업은 자영업 판매 서비스직 일 차 산업 순으로 나타났다

예시 flac: train_data_01/003/181/181_003_2361.flac


In [3]:
# AUDIO_INFO: 화자 메타 형식 확인 (데이터셋마다 포맷이 달라 우선 원본 확인)
ai = ROOT / "AUDIO_INFO"
if ai.exists():
    print(ai.read_text(encoding="utf-8", errors="replace")[:1500])
else:
    print("AUDIO_INFO 없음")

SPEAKERID|NAME|SEX|SCRIPTID|DATASET
104|BaekGeonJong|m|003|test_data_01
105|ChoSoYeon|f|003|test_data_01
106|KimJiWon|f|003|train_data_01
107|KimNamHyung|m|003|train_data_01
108|KimYeLim|f|003|train_data_01
109|LeeSeoJeong|f|003|train_data_01
110|LucasJo|m|003|train_data_01
111|MoonJinSil|f|003|train_data_01
112|ParkSeHyeon|m|003|test_data_01
113|ParkSeulGi|f|003|train_data_01
114|MichaelChoi|m|003|train_data_01
115|JangJeongHyeon|f|003|train_data_01
116|JungYoungHoon|m|003|train_data_01
117|KwonBumKi|m|003|train_data_01
118|LeeSeungMin|m|003|test_data_01
119|LimHwiRang|f|003|train_data_01
120|OhHeeSu|m|003|train_data_01
121|HanHeeWon|f|003|test_data_01
122|JeongDaUn|f|003|train_data_01
123|JeongHaeSu|f|003|train_data_01
124|JoAnHyeop|m|003|train_data_01
125|JungWooJun|m|003|train_data_01
126|KwackHyeYeong|f|003|test_data_01
127|LeeYoungEun|f|003|train_data_01
128|ParkJiHye|f|003|train_data_01
129|YoonChangHee|m|003|train_data_01
130|ChoiHyeJin|f|003|train_data_01
131|ChoiSeungWon|f|00

In [4]:
# manifest 빌드: 모든 .trans.txt 파싱 → (split, utt_id, speaker, text, flac_path)
def build_rows(split_dir, split):
    rows = []
    for trans in sorted(Path(split_dir).rglob("*.trans.txt")):
        d = trans.parent
        for line in trans.read_text(encoding="utf-8", errors="replace").splitlines():
            parts = line.strip().split(maxsplit=1)   # 'utt_id  전사' (공백/탭 모두 처리)
            if len(parts) < 2:
                continue
            uid, text = parts[0], parts[1].strip()
            rows.append((split, uid, uid.split("_")[0], text, str(d / f"{uid}.flac")))
    return rows

rows = build_rows(TRAIN, "train") + build_rows(TEST, "test")
df = pd.DataFrame(rows, columns=["split", "utt_id", "speaker", "text", "flac_path"])
df["text"] = df["text"].astype("object")   # pyarrow string 회피 (Python re 사용)

print("split별 발화 수:")
print(df["split"].value_counts().to_string())
print(f"\n총 {len(df):,} 발화 / 화자 {df['speaker'].nunique()}명")
df.head()

split별 발화 수:
split
train    22263
test       457

총 22,720 발화 / 화자 115명


,split,utt_id,speaker,text,flac_path
0,train,106_003_0077,106,인사를 결정하는 과정에서 당 지도부가 우 원내대표 및 원내지도부와 충분한 상의를 거...,/data/ASR/RAW/OpenSLR_SLR40_Zeroth_ko/train_da...
1,train,106_003_0098,106,특히 전 전 대통령은 내빈 소개 때 여전히 전두환 각하라고 불리기도 했다,/data/ASR/RAW/OpenSLR_SLR40_Zeroth_ko/train_da...
2,train,106_003_0161,106,건보공단이 이씨에게 징수금을 부과하고 재산을 압류한 것은 부당하다고 판결했다,/data/ASR/RAW/OpenSLR_SLR40_Zeroth_ko/train_da...
3,train,106_003_0199,106,아버지 신격호 롯데 총괄회장이 축출한 가신들을 결집시킨 신동빈 회장이 한 일 롯데의...,/data/ASR/RAW/OpenSLR_SLR40_Zeroth_ko/train_da...
4,train,106_003_0238,106,앵커 오늘 새벽 전남 여수에서 오늘 새벽 전남 여수에서 무궁화호 열차가 탈선했습니다,/data/ASR/RAW/OpenSLR_SLR40_Zeroth_ko/train_da...


In [5]:
# 매칭/결측 점검
chk = df.sample(min(300, len(df)), random_state=0)
miss = int(chk["flac_path"].map(lambda p: not Path(p).exists()).sum())
print(f"표본 {len(chk)}건 중 flac 없음: {miss}")
print(f"빈 전사: {int((df['text'].str.strip()=='').sum())}건")
print(f"중복 전사: {int(df['text'].duplicated().sum())}건")

표본 300건 중 flac 없음: 0
빈 전사: 0건
중복 전사: 19716건


## 2. 분포

In [6]:
df["text_len"] = df["text"].str.len()
g = df.groupby("split")
print("=== split별 ===")
print(pd.DataFrame({
    "발화수":     g.size(),
    "화자수":     g["speaker"].nunique(),
    "글자수평균": g["text_len"].mean().round(1),
    "글자수중앙": g["text_len"].median(),
    "최장":       g["text_len"].max(),
}).to_string())

=== split별 ===
         발화수  화자수  글자수평균  글자수중앙  최장
split                              
test     457   10   55.7   52.0  98
train  22263  105   55.5   53.0  99


## 3. 전사 컨벤션

In [7]:
txt = df["text"].fillna("")

print("특수문자 전수 (상위 30):")
print(txt.str.findall(r"[^가-힣a-zA-Z0-9\s]").explode().value_counts().head(30).to_string())

print(f"\n영문 포함: {txt.str.contains(r'[A-Za-z]').mean()*100:.2f}%")
print(f"숫자 포함: {txt.str.contains(r'[0-9]').mean()*100:.2f}%")

# KsponSpeech식 태그/이중전사가 있는지 점검 (낭독체면 0일 것)
print("\nKsponSpeech식 주석 점검:")
found = False
for tag in ["b/", "n/", "l/", "o/", "u/", ")/("]:
    c = int(txt.str.contains(re.escape(tag)).sum())
    if c:
        print(f"  '{tag}' 출현: {c:,}건"); found = True
if not found:
    print("  없음 → 깨끗한 낭독체 전사. 정규화는 구두점/공백 정도면 충분.")

특수문자 전수 (상위 30):
Series([], )

영문 포함: 0.00%
숫자 포함: 0.00%

KsponSpeech식 주석 점검:
  없음 → 깨끗한 낭독체 전사. 정규화는 구두점/공백 정도면 충분.


## 3-1. 실제 음성 청취

In [8]:
# 샘플 청취 (전사와 함께) — FLAC은 헤더가 있어 바로 재생
print("랜덤 샘플 3개:")
for r in df.sample(3, random_state=1).itertuples():
    print(f"[{r.split}] {r.speaker} | {r.text[:70]}")
    display(play_flac(r.flac_path))

랜덤 샘플 3개:
[train] 191 | 그게 한 천 육백 억 원이 넘는 것으로 그렇게 알려지고 있습니다


[train] 194 | 이런 조언을 들었는데 여기에다가 한마디를 더 한 게 매서운데요


[train] 114 | 한국 무속의 경우에도 남쪽 지방과 북쪽 지방이 상당히 다르다


## 4. 오디오 속성 (FLAC)

> FLAC은 헤더가 있어 sr/채널/비트/길이를 **직접 읽음**. KsponSpeech처럼 가정·추정할 필요 없음.

In [9]:
# FLAC 헤더에서 규격 직접 읽기
samp = df.sample(min(2000, len(df)), random_state=3)
samp = samp[samp["flac_path"].map(lambda p: Path(p).exists())]
info = samp["flac_path"].map(flac_info)
sr  = info.map(lambda x: x[0])
ch  = info.map(lambda x: x[1])
sub = info.map(lambda x: x[2])
dur = info.map(lambda x: x[3])

print("sample_rate :", sr.value_counts().to_dict())
print("channels    :", ch.value_counts().to_dict())
print("subtype(비트):", sub.value_counts().to_dict())
print(f"\n길이(초): 평균 {dur.mean():.2f} / 중앙 {dur.median():.2f} / p95 {dur.quantile(.95):.2f} / 최대 {dur.max():.2f}")
print(f"표본 평균 × 전체 = 총시간 추정: {dur.mean()*len(df)/3600:.1f} 시간")

sample_rate : {16000: 2000}
channels    : {1: 2000}
subtype(비트): {'PCM_16': 2000}

길이(초): 평균 8.27 / 중앙 7.74 / p95 12.98 / 최대 20.67
표본 평균 × 전체 = 총시간 추정: 52.2 시간


## 5. 화자 분리 (speaker-disjoint)

> Zeroth는 utt_id에 화자 ID가 있어 KsponSpeech와 달리 화자 누수 점검이 가능.

In [10]:
tr_spk = set(df[df.split == "train"]["speaker"])
te_spk = set(df[df.split == "test"]["speaker"])
overlap = tr_spk & te_spk
print(f"train 화자 {len(tr_spk)}명 / test 화자 {len(te_spk)}명")
print(f"train ∩ test 겹치는 화자: {len(overlap)}명")
print("→ 0이면 speaker-disjoint = 평가셋으로 적합 (새 화자 일반화 측정 가능)")

print("\n화자별 발화 수 분포:")
spc = df["speaker"].value_counts()
print(f"  화자당 평균 {spc.mean():.0f} / 중앙 {spc.median():.0f} / 최대 {spc.max()} / 최소 {spc.min()}")

train 화자 105명 / test 화자 10명
train ∩ test 겹치는 화자: 0명
→ 0이면 speaker-disjoint = 평가셋으로 적합 (새 화자 일반화 측정 가능)

화자별 발화 수 분포:
  화자당 평균 198 / 중앙 104 / 최대 763 / 최소 3


In [11]:
# ============================================================
# 비식별화(PII) 토큰 포함 발화 → 전사 + 음성 직접 청취  (모든 EDA 노트북 공용)
# 파서로 DataFrame을 만든 셀을 먼저 실행한 뒤, 이 셀을 새 셀에 붙여 실행.
# DataFrame 변수(df/df_v/...)와 오디오 경로 컬럼(audio_path/wav/...)을 자동 탐지.
# ============================================================
import re
from pathlib import Path
from IPython.display import Audio, display
try:
    import soundfile as sf
except ImportError:
    sf = None
    print("⚠ soundfile 없음 → conda activate TRAIN-ASR")

import pandas as pd

# ---------- 설정 ----------
N_LISTEN = 5          # 들어볼 발화 수
RANDOM_STATE = 0      # None이면 매번 다른 표본
# 비식별화(PII) 토큰: 음향 토큰(b/ n/ l/ o/ u/)과 구분되는 익명화 전용 패턴
PII_PATTERNS = {
    "@ (이름 마커)":           re.compile(r"@"),
    "ㅇㅇ류 (익명화 2자+)":     re.compile(r"ㅇ{2,}"),
    "name/ (이름 태그)":        re.compile(r"(?:^|\s)name/", re.I),
    "[마스킹]":                re.compile(r"\[[^\]]{0,15}\]"),
    "<마스킹>":                re.compile(r"<[^>]{0,15}>"),
    "*** (별표 2+)":           re.compile(r"\*{2,}"),
    "xxx (엑스 2+)":           re.compile(r"[xX]{2,}"),
    "○○ (공백원 2+)":          re.compile(r"[○◯]{2,}"),
}

# ---------- 1) text DataFrame 자동 탐지 ----------
def _find_text_frame():
    g = globals()
    for name in ["df_v","df_valid","df","df_t","df_train","d","a"]:
        o = g.get(name)
        if isinstance(o, pd.DataFrame) and "text" in o.columns:
            return name, o
    for name, o in g.items():
        if not name.startswith("_") and isinstance(o, pd.DataFrame) and "text" in o.columns:
            return name, o
    return None, None

_name, _df = _find_text_frame()
if _df is None:
    raise RuntimeError("text 컬럼 DataFrame 없음 — 파서 셀을 먼저 실행하세요.")

# ---------- 2) 오디오 경로 컬럼 자동 탐지 ----------
AUDIO_COL = next((c for c in ["audio_path","wav","src_wav","audio","path","filepath"]
                  if c in _df.columns), None)
print(f"[대상] DataFrame '{_name}' · {len(_df):,} 발화 · 오디오 컬럼: {AUDIO_COL or '없음(PCM 직접 노트북일 수 있음)'}\n")

# ---------- 3) PII 토큰 집계 ----------
_txt = _df["text"].fillna("").astype("object")
print("=== 비식별화 토큰 집계 (text 원본) ===")
present = []
for label, pat in PII_PATTERNS.items():
    occ = int(_txt.str.count(pat).sum())
    utt = int(_txt.str.contains(pat).sum())
    if occ:
        present.append((label, pat, utt, occ))
        print(f"  {label:20s} 발화 {utt:>6,} · 출현 {occ:>6,} ({utt/len(_df)*100:.3f}%)")
if not present:
    print("  → 이 데이터셋엔 정의된 비식별화 토큰이 없음 (자유발화/낭독 등). 청취 생략.")

# ---------- 4) PII 포함 발화만 필터 → 전사 + 음성 재생 ----------
if present and AUDIO_COL:
    mask = pd.Series(False, index=_df.index)
    for _, pat, _, _ in present:
        mask |= _txt.str.contains(pat)
    hits = _df[mask]
    print(f"\n=== 비식별화 토큰 포함 발화 {len(hits):,}건 중 {min(N_LISTEN,len(hits))}개 청취 ===")
    print("   (전사의 마스킹 부분이 음성에서 실제로 어떻게 발화되는지 직접 확인)\n")
    sample = hits.sample(min(N_LISTEN, len(hits)), random_state=RANDOM_STATE)
    for r in sample.itertuples():
        text = getattr(r, "text", "")
        ap = getattr(r, AUDIO_COL, None)
        # 어떤 PII 패턴에 걸렸는지 표시
        tags = [lab for lab, pat, _, _ in present if pat.search(text or "")]
        print(f"[{', '.join(tags)}]")
        print(f"  전사: {text[:100]}")
        if sf and ap and Path(str(ap)).exists():
            try:
                data, sr = sf.read(str(ap))
                display(Audio(data, rate=sr))
            except Exception as e:
                print(f"   (재생 실패: {str(e)[:50]})")
        else:
            print(f"   (오디오 경로 없음/미존재: {ap})")
        print()
elif present and not AUDIO_COL:
    print("\n⚠ PII 토큰은 있으나 오디오 경로 컬럼을 못 찾음.")
    print("  이 노트북이 PCM을 직접 읽는 방식이면, 아래처럼 수동 지정:")
    print("  → sample = _df[mask].sample(N_LISTEN); 각 행의 키로 원본 PCM 경로를 구성해 재생")

[대상] DataFrame 'df' · 22,720 발화 · 오디오 컬럼: 없음(PCM 직접 노트북일 수 있음)

=== 비식별화 토큰 집계 (text 원본) ===
  → 이 데이터셋엔 정의된 비식별화 토큰이 없음 (자유발화/낭독 등). 청취 생략.


In [15]:
import json
import pandas as pd
import jiwer
from pathlib import Path

REPO = Path("/home/kiho/workspace/TRAIN-ASR")        # 절대경로 — 작업 위치 무관
BENCH_ID = "OpenSLR_SLR40_Zeroth_ko_general_clean"
RESULT = REPO / "BENCHMARK" / "results" / f"whisper_small__{BENCH_ID}"

# predictions.jsonl 자동 탐색 (하위 폴더 구조가 달라도 찾음)
cands = list(RESULT.rglob("predictions.jsonl"))
if not cands:
    raise FileNotFoundError(f"predictions.jsonl 못 찾음 — RESULT 확인: {RESULT} (존재: {RESULT.exists()})")
pred_path = cands[0]
print(f"로드: {pred_path}")
df = pd.DataFrame(json.loads(l) for l in pred_path.read_text(encoding="utf-8").splitlines() if l)

# 1) 발화별 CER
df["cer"] = [jiwer.cer(r, h) * 100 if r else float("nan")
             for r, h in zip(df["text_normalized"], df["prediction_normalized"])]

# 2) 글자 집합 Jaccard 유사도 (공백 제거) — 정렬오류면 거의 0
def char_jaccard(a, b):
    sa, sb = set(a.replace(" ", "")), set(b.replace(" ", ""))
    if not sa or not sb:
        return 0.0
    return len(sa & sb) / len(sa | sb)

df["jaccard"] = [char_jaccard(r, h)
                 for r, h in zip(df["text_normalized"], df["prediction_normalized"])]

# 3) 발화 속도 (정답 글자수 / 오디오 길이) — 정답이 오디오보다 길면 비정상적으로 높음
df["chars_per_sec"] = df["text_normalized"].str.replace(" ", "").str.len() / df["duration_sec"]

print(f"전체 {len(df)}건 | 평균 CER {df['cer'].mean():.1f}% | 한국어 정상 발화속도 ~4-7 글자/초")

로드: /home/kiho/workspace/TRAIN-ASR/BENCHMARK/results/whisper_small__OpenSLR_SLR40_Zeroth_ko_general_clean/OpenSLR_SLR40_Zeroth_ko_general_clean/predictions.jsonl
전체 457건 | 평균 CER 10.2% | 한국어 정상 발화속도 ~4-7 글자/초


In [16]:
# 정렬오류 의심: 글자 겹침이 거의 없는데(jaccard 낮음) CER 매우 높음
# → 모델은 멀쩡히 받아썼는데 정답과 전혀 다른 문장 = 정답 전사가 그 오디오 것이 아닐 가능성
suspect = df[(df["jaccard"] < 0.15) & (df["cer"] > 50)].sort_values("jaccard")
print(f"⚠️ 정렬오류 의심 후보: {len(suspect)}건 / 전체 {len(df)}건\n")
for _, r in suspect.iterrows():
    print(f"[jaccard {r.jaccard:.2f} | CER {r.cer:.0f}% | {r.duration_sec:.1f}초 | "
          f"발화속도 {r.chars_per_sec:.1f}자/초] {Path(r.audio).name}")
    print(f"  정답: {r.text_normalized}")
    print(f"  예측: {r.prediction_normalized}\n")

⚠️ 정렬오류 의심 후보: 1건 / 전체 457건

[jaccard 0.08 | CER 88% | 20.5초 | 발화속도 2.4자/초] 105_003_0478.wav
  정답: 차 문이 종잇장처럼 얇지 않으니 문 두께 스물 센티미터를 빼면 실제 승 하차 여유 공간은 스물 센티미터라는 계산이 나옵니다
  예측: 계속 찍는 게 녹음되고 있는 거야 조용히 하면



In [17]:
from IPython.display import Audio, display


for _, r in suspect.iterrows():
    print("="*70)
    print(f"파일: {Path(r.audio).name}  ({r.duration_sec:.1f}초)")
    print(f"  정답(전사): {r.text_normalized}")
    print(f"  예측(모델): {r.prediction_normalized}")
    display(Audio(r.audio))   
    print()

파일: 105_003_0478.wav  (20.5초)
  정답(전사): 차 문이 종잇장처럼 얇지 않으니 문 두께 스물 센티미터를 빼면 실제 승 하차 여유 공간은 스물 센티미터라는 계산이 나옵니다
  예측(모델): 계속 찍는 게 녹음되고 있는 거야 조용히 하면
